# QELM on the measurement record M, sweep over the sample-to-parameter ratio

Experiment `experiments/QELM_M/exp_QELM_M_ratio.py` (issue #96), on top of `experiments/QELM_M/exp_QELM_M.py` (issue #94). The reservoir, the measurements and the target are those of `02_QELM_M_analysis.ipynb`: a disordered TFIM with an ancilla entangled with the charge of the system, the system qubits measured in Z with probability `p` after each of the `nbr_time_evolution` steps, the record `M` as features (0 for an unmeasured qubit) and `<Z_0>` of the ancilla at the final time as target.

Here `N` is fixed, so the number of weights of the ridge readout is fixed too:

`n_params = nbr_time_evolution * N + 1`

(the flattened record and the bias column; 16 at `N = 3`, whatever `p`). What changes is the number of training samples, through the ratio

`alpha_ratio = n_train / n_params`.

`alpha_ratio < 1` is the underdetermined regime: fewer training equations than weights, and only the ridge penalty `training.alpha > 0` makes the fit unique. `alpha_ratio > 1` is the overdetermined regime. At `alpha_ratio = 1`, the interpolation threshold, a weakly regularized readout fits the training targets exactly, noise included, and the test error peaks (double descent). The ratio is the sample ratio only: it is `experiment.alpha_ratio`, and the ridge penalty `training.alpha` stays an independent key.

The ratio counts the training samples, not all `L`, since only those constrain the weights. The test set has a fixed size `n_test` for every ratio, so the test NMSE of two ratios is measured on as many samples.

What to look for: at `p = 1` the record fixes the charge, `<Z_0> = ±1` is a function of it (issue #104), and the test NMSE keeps falling with the ratio. At `p < 1` part of the record is missing, `<Z_0>` is no longer a function of the features, and the NMSE saturates at a floor set by the information the record holds: the gap between the curves at large ratio measures what the measurements reveal about the ancilla, the quantity that the measurement-induced transition controls.

The last section draws the same sweep as heatmaps over the 2D grid of `p` and the ratio (issue #164), one per metric, to show where the measurement-induced transition (along `p`) and the finite-sampling, double-descent effects (along the ratio) cross over.

**Qubit counts (issue #235).** `N` is the number of system qubits, the ancilla excluded; the simulated state has `N_total = N + 1` qubits. The sizes of this notebook, its defaults and the results quoted in its text were moved to this convention: a result obtained at `N = 6` before issue #235, when `N` counted the ancilla, is quoted here at `N = 5`, the same system.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here, or through the environment variable of the same name (this is what the test `tests/test_qelm_m_ratio_notebook.py` does for a small sweep). Notation, the same in every notebook: `N` is the number of system qubits, ancilla excluded (issue #235), and `L` the number of samples per run; here `L = n_train + N_TEST`, set by the ratio, so it is not a parameter. `ALPHA` is the ridge penalty `training.alpha`, not the ratio. Each sweep goes to this notebook's own folder, `outputs/QELM_M_ratio/<date_time>/`, and the notebook reads back exactly that folder (issue #151). `RUN = False` reads back the latest sweep of this folder without launching a new one.

A run that crashes (a segmentation fault is exit code 139) is tried again, then listed in `failed_runs.txt` in the sweep folder, and the sweep goes on: the cells below draw the runs that finished, a blank cell or a smaller `n_runs` marks a missing one. `RESUME` set to the path of an earlier sweep folder completes that sweep instead of starting a new one: the runs already done are skipped and the failed or missing ones are launched again (issue #168).

The default grid is the one of the heatmaps (issue #164): `p` from 0 to 1 by steps of 0.1, and the ratio from 0.25 to 10, spaced logarithmically a factor about `sqrt(2)` apart, with `1.0` among them so that the interpolation threshold is a row of the grid. That is `11 x 12` points per seed and `N`, about 2 s per run. The curves against the ratio show only the values of `CURVE_P_VALUES` that the sweep has, to stay readable.

In [ ]:
N_VALUES = os.environ.get("N_VALUES", "3")  # system qubits N, ancilla excluded
SEEDS = os.environ.get("SEEDS", "42 43 44")
# Measurement probability, 0 to 1 (the y axis of the heatmaps).
P_VALUES = os.environ.get("P_VALUES", "0.0 0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1.0")
# The values of p drawn as curves against the ratio.
CURVE_P_VALUES = os.environ.get("CURVE_P_VALUES", "0.2 0.5 1.0")
# Training samples per weight, n_train / n_params (experiment.alpha_ratio),
# 0.25 to 10 on a log scale.
ALPHA_RATIOS = os.environ.get(
    "ALPHA_RATIOS", "0.25 0.35 0.5 0.71 1.0 1.41 2.0 2.83 4.0 5.66 8.0 10.0"
)
N_TEST = os.environ.get("N_TEST", "200")  # test samples per run
ALPHA = os.environ.get("ALPHA", "1e-3")  # ridge penalty (training.alpha)
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "1") == "1"
# Sweep folder to complete (runs already done are skipped); "" for a new sweep.
RESUME = os.environ.get("RESUME", "")

## Launch the sweep

In [ ]:
from QELM.utils import latest_sweep, new_sweep_dir, run_with_progress

# This notebook's own output folder (issue #151).
EXPERIMENT = "QELM_M_ratio"

if RUN:
    # Create the sweep folder here and hand it to the launcher (OUT_DIR):
    # the cells below read this sweep and no other.
    # RESUME: complete that sweep, the runs already done are skipped.
    sweep_dir = Path(RESUME) if RESUME else new_sweep_dir(OUT_ROOT, EXPERIMENT)
    env = {
        **os.environ,
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "P_VALUES": P_VALUES,
        "ALPHA_RATIOS": ALPHA_RATIOS,
        "N_TEST": N_TEST,
        "ALPHA": ALPHA,
        "OUT_DIR": str(sweep_dir),
    }
    n_runs = (
        len(N_VALUES.split())
        * len(SEEDS.split())
        * len(P_VALUES.split())
        * len(ALPHA_RATIOS.split())
    )
    # One bar for the whole sweep, one step per run. A run that keeps failing
    # is listed in failed_runs.txt and the sweep goes on.
    run_with_progress(
        ["bash", "experiments/QELM_M/exp_QELM_M_ratio.sh"], total=n_runs, env=env
    )
    failed = sweep_dir / "failed_runs.txt"
    if failed.is_file():
        print(
            f"WARNING: failed runs, missing from the figures:\n{failed.read_text()}"
            f'Set RESUME = "{sweep_dir}" and run again to retry them.'
        )
else:
    # RUN = False: the latest sweep of this notebook only.
    sweep_dir = latest_sweep(OUT_ROOT / EXPERIMENT)
print(f"sweep: {sweep_dir}")

## Read back the metrics

The ratio of each run is computed from its `metrics.json`, as `n_train / n_features`: the ratio actually used, after rounding `alpha_ratio * n_params` to a whole number of samples. `samples.L` is not read from `.hydra/config.yaml`, which Hydra writes before the script sets the sample count; the sizes used are in `metrics.json` and `config_used.yaml`.

In [ ]:
from QELM.utils import load_sweep, summarize

# (key in metrics.json, label, log scale).
METRICS = [
    ("nmse", "NMSE (test)", True),
    ("r2", "R2 (test)", False),
    ("balanced_accuracy", "balanced accuracy of sign <Z_0> (test)", False),
]
rows = load_sweep(
    sweep_dir,
    keys=(("N", "N"), ("seed", "seed"), ("p", "projective_measurement.p")),
)
for r in rows:
    r["ratio"] = r["n_train"] / r["n_features"]
# The heatmaps (last section) also show the accuracy.
table = summarize(
    rows, by=("N", "p", "ratio"), metrics=[k for k, _, _ in METRICS] + ["accuracy"]
)
print(f"{len(rows)} runs, n_params = {sorted({r['n_features'] for r in rows})}")
print(
    f"{'N':>3} {'p':>5} {'ratio':>6} {'runs':>4} "
    + " ".join(f"{key:>20}" for key, _, _ in METRICS)
)
for r in table:
    values = " ".join(
        f"{r[f'{key}_mean']:>9.3g} ± {r[f'{key}_sem']:<8.2g}" for key, _, _ in METRICS
    )
    print(f"{r['N']:>3} {r['p']:>5.2f} {r['ratio']:>6.3g} {r['n_runs']:>4} {values}")

## Metrics against the ratio

One row of panels per `N` (the first `N` on top), one panel per metric, one curve per measurement probability `p` of `CURVE_P_VALUES`, the mean over the seeds with the standard error as error bars. Each `N` has its own panels, so its own axis ranges, and its own ratios: `n_params` depends on `N`, and so does the rounding of `alpha_ratio * n_params` to a whole number of samples. The ratio axis is logarithmic, and so is the NMSE axis, which spans several orders of magnitude. The dashed vertical line marks the interpolation threshold `alpha_ratio = 1`, where the training system goes from underdetermined to overdetermined; on the balanced accuracy, the dotted gray line marks chance, 0.5.

The figure and the data it shows are saved in the sweep folder, as `metrics_vs_alpha_ratio.png` and `metrics_vs_alpha_ratio.json` (`QELM.utils.plots.save_figure`).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter

from QELM.utils.plots import (
    color_map,
    draw_curve,
    save_figure,
    sizes,
    subplots,
    sweep_curve,
)

n_values = sizes(table)


def ratio_ticks(ax, ratios):
    """Label the log ratio axis at the powers of 2 within the swept ratios.

    Matplotlib's default labels on a log axis spanning less than two decades
    (0.25 to 10) overlap.
    """
    ticks = [2.0**k for k in range(-6, 7) if min(ratios) <= 2.0**k <= max(ratios)]
    ax.set_xticks(ticks, labels=[f"{t:g}" for t in ticks])
    ax.xaxis.set_minor_formatter(NullFormatter())


# One row of panels per N.
fig, axes = subplots(
    len(n_values),
    len(METRICS),
    figsize=(6.5 * len(METRICS), 5.5 * len(n_values)),
    squeeze=False,
)
ps = sorted({r["p"] for r in table if r["p"] in map(float, CURVE_P_VALUES.split())})
if not ps:  # none of CURVE_P_VALUES was swept: draw every p
    ps = sorted({r["p"] for r in table})
colors = color_map([f"p = {p:g}" for p in ps])
for row, N in zip(axes, n_values, strict=True):
    for p in ps:
        rows_p = [r for r in table if r["p"] == p]
        label = f"p = {p:g}"
        for (key, _, log), ax in zip(METRICS, row, strict=True):
            curve = sweep_curve(rows_p, N, "ratio", key)
            draw_curve(ax, curve, colors[label], label=label, log=log)
    for (key, label, _), ax in zip(METRICS, row, strict=True):
        ax.axvline(1.0, color="black", linestyle="--", label="interpolation threshold")
        if key == "balanced_accuracy":
            ax.axhline(0.5, color="gray", linestyle=":", label="chance")
        ax.set_xscale("log")
        ratio_ticks(ax, [r["ratio"] for r in table if r["N"] == N])
        ax.set(
            xlabel="alpha_ratio = n_train / n_params",
            ylabel=label,
            title=f"{label}, N = {N}",
        )
        ax.legend()
fig.suptitle("QELM on M: metrics against the sample-to-parameter ratio")
figure_path = save_figure(
    fig, sweep_dir, "metrics_vs_alpha_ratio", data={"table": table}
)
print(f"figure: {figure_path}")
plt.show()

## Heatmaps over p and the ratio

Issues #164 and #167. The same sweep, now as one color map per metric: the mean over the seeds at each point of the grid, `alpha_ratio` on a logarithmic x axis and `p` on the y axis, one row of panels per `N` (the first `N` on top), each with its own grid of ratios and, for the NMSE, its own color range. Each grid point is the center of its cell, with geometric cell edges on the ratio axis (`QELM.utils.plots.cell_edges`), so the cells have the same width on screen. The dashed white vertical line marks the interpolation threshold `alpha_ratio = 1`. A blank cell is a point without a run.

The two axes carry two different effects: along `p`, the measurement-induced transition, how much the record `M` reveals about `<Z_0>`; along the ratio, finite sampling, with the double-descent peak at `alpha_ratio = 1` and the fall towards the floor set by the information in `M` above it. The map shows the ratio needed to reach the floor at each `p`, and whether the peak depends on `p`.

The color scale of each metric:

- NMSE: logarithmic, since it spans several orders of magnitude;
- R2: from -1 to 1. R2 is at most 1, and the peak at `alpha_ratio = 1` can drive it far below 0; the cells below -1 (a readout more than twice as bad as the mean of the target) take the bottom color, so the rest of the map keeps its contrast;
- accuracy and balanced accuracy of the sign of `<Z_0>`: from 0.5 (chance for the balanced accuracy) to 1.

The figure and the grids it shows are saved in the sweep folder, as `heatmaps_p_alpha_ratio.png` and `heatmaps_p_alpha_ratio.json` (each panel's `heatmaps` entry holds the cell edges and the values).

In [ ]:
from QELM.utils.plots import draw_heatmap, sweep_grid

# (key in metrics.json, label, log color scale, bottom and top of the scale).
HEATMAPS = [
    ("nmse", "NMSE (test)", True, None, None),
    ("r2", "R2 (test)", False, -1.0, 1.0),
    ("accuracy", "accuracy of sign <Z_0> (test)", False, 0.5, 1.0),
    ("balanced_accuracy", "balanced accuracy of sign <Z_0> (test)", False, 0.5, 1.0),
]
fig_maps, axes_maps = subplots(
    len(n_values),
    len(HEATMAPS),
    figsize=(5.5 * len(HEATMAPS), 4.8 * len(n_values)),
    squeeze=False,
)
grids = {}
for row, N in zip(axes_maps, n_values, strict=True):
    for (key, label, log_color, vmin, vmax), ax in zip(HEATMAPS, row, strict=True):
        grid = sweep_grid(table, "ratio", "p", key, size=N)
        grids[(N, key)] = grid
        draw_heatmap(
            ax, grid, label=label, log_x=True, log_color=log_color, vmin=vmin, vmax=vmax
        )
        ratio_ticks(ax, grid.x)
        ax.axvline(1.0, color="white", linestyle="--", label="interpolation threshold")
        ax.set(
            xlabel="alpha_ratio = n_train / n_params",
            ylabel="measurement probability p",
            title=f"{label}, N = {N}",
        )
fig_maps.suptitle("QELM on M: metrics over p and the sample-to-parameter ratio")
heatmap_path = save_figure(
    fig_maps, sweep_dir, "heatmaps_p_alpha_ratio", data={"table": table}
)
print(f"figure: {heatmap_path}")
plt.show()